# Wiki Movie Plots Dataset
## Data Preprocessing & Cleaning

produce a clean version of the dataset by removing
1. **Missing values** (rows with any `NaN`)
2. **Duplicate records** (exact duplicate rows + duplicate titles)
3. **`Unknown` records** (where the column value literally is `Unknown`/`unknown`)

> Note: `Title`, `Plot` and `Wiki Page` may contain the word "unknown" inside
> legitimate text (e.g. the movie *The Unknown*). Those are **not** removed —
> only values that are exactly `Unknown` in categorical columns are.


## 1. Setup & Load

In [1]:
import pandas as pd
import numpy as np

DATA_PATH = "../data/wiki_movie_plots_deduped.csv"
df = pd.read_csv(DATA_PATH)
print(f"Original shape: {df.shape[0]:,} rows x {df.shape[1]} columns")

Original shape: 34,886 rows x 8 columns


## 2. Inspect Missing & Unknown Values

In [2]:
print("=== Missing values (NaN) per column ===")
print(df.isna().sum()[df.isna().sum() > 0].to_string())
print(f"\nRows with at least one missing value: {df.isna().any(axis=1).sum():,}")

=== Missing values (NaN) per column ===
Cast    1422

Rows with at least one missing value: 1,422


In [3]:
print("=== Literal 'Unknown' values per column ===")
for col in df.columns:
    if df[col].dtype == "str":
        n_unknown = (df[col] == "Unknown").sum() + (df[col] == "unknown").sum()
        if n_unknown > 0:
            print(f"{col}: {n_unknown:,}")

=== Literal 'Unknown' values per column ===
Title: 2
Director: 1,124
Cast: 1
Genre: 6,083


## 3. Check Duplicates

In [4]:
# 1) Exact duplicate rows (all columns identical)
exact_dups = df.duplicated().sum()
print(f"Exact duplicate rows: {exact_dups:,}")

# 2) Duplicate titles (same movie title)
title_dups = df["Title"].duplicated().sum()
print(f"Rows with a duplicated Title: {title_dups:,}")

print(f"\nUnique titles: {df['Title'].nunique():,} of {len(df):,} rows")

Exact duplicate rows: 0
Rows with a duplicated Title: 2,454

Unique titles: 32,432 of 34,886 rows


In [5]:
# Show example of duplicated titles
dup_titles = df[df["Title"].duplicated(keep=False)].sort_values("Title")
print(f"Example duplicate titles ({len(dup_titles)} rows involved):")
print(dup_titles[["Title", "Release Year", "Origin/Ethnicity", "Director"]].head(12).to_string())

Example duplicate titles (4525 rows involved):
              Title  Release Year Origin/Ethnicity                   Director
17813         $9.99          2009       Australian            Tatia Rosenthal
17796         $9.99          2008       Australian            Tatia Rosenthal
34228            10          2014          Russian                    Unknown
9556             10          1979         American              Blake Edwards
17168   100 Streets          2017         American               Jim O'Hanlon
21611   100 Streets          2016          British     Director: Jim O'Hanlon
24073     100% Love          2012          Bengali                Rabi Kinagi
32475     100% Love          2011           Telugu                    Sukumar
34130            12          2007          Russian           Nikita Mikhalkov
32245            12          2006           Telugu                      Style
32273            12          2006           Telugu  Tata Birla Madhyalo Laila
6700   12 Angry M

## 4. Clean the Dataset

In [6]:
before = len(df)

# 1) Remove exact duplicate rows (keep first occurrence)
df = df.drop_duplicates()
print(f"After removing exact duplicates: {len(df):,} (removed {before - len(df):,})")

After removing exact duplicates: 34,886 (removed 0)


In [7]:
# 2) Remove duplicate titles (keep first occurrence)
before2 = len(df)
df = df.drop_duplicates(subset=["Title"], keep="first")
print(f"After removing duplicate titles: {len(df):,} (removed {before2 - len(df):,})")

After removing duplicate titles: 32,432 (removed 2,454)


In [8]:
# 3) Remove rows with any missing value
before3 = len(df)
df = df.dropna()
print(f"After removing missing-value rows: {len(df):,} (removed {before3 - len(df):,})")

After removing missing-value rows: 31,130 (removed 1,302)


In [9]:
# 4) Remove rows where categorical columns literally equal 'Unknown'/'unknown'
before4 = len(df)

unknown_mask = (
    df["Director"].isin(["Unknown"])
    | df["Genre"].isin(["unknown"])
    | df["Cast"].isin(["Unknown"])
)
print(f"Rows with an 'Unknown' value: {unknown_mask.sum():,}")
df = df[~unknown_mask]

print(f"After removing 'Unknown' records: {len(df):,} (removed {before4 - len(df):,})")

Rows with an 'Unknown' value: 5,154
After removing 'Unknown' records: 25,976 (removed 5,154)


## 5. Final Check & Summary

In [10]:
print(f"=== CLEAN DATASET ===")
print(f"Final shape: {df.shape[0]:,} rows x {df.shape[1]} columns")

print("\nMissing values remaining:", df.isna().sum().sum())
print("Exact duplicates remaining:", df.duplicated().sum())
print("Duplicate titles remaining:", df["Title"].duplicated().sum())
print("'Unknown' values remaining:",
      (df["Director"] == "Unknown").sum()
      + (df["Genre"] == "unknown").sum()
      + (df["Cast"] == "Unknown").sum())

=== CLEAN DATASET ===
Final shape: 25,976 rows x 8 columns

Missing values remaining: 0


Exact duplicates remaining: 0
Duplicate titles remaining: 0
'Unknown' values remaining: 0


In [11]:
print(f"Rows removed overall: {34_886 - len(df):,} ({100 * (34_886 - len(df)) / 34_886:.1f}%)")
print(df.head().to_string())

Rows removed overall: 8,910 (25.5%)
    Release Year                     Title Origin/Ethnicity                                 Director                               Cast         Genre                                                    Wiki Page                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                  

## 6. Save Cleaned Dataset

In [12]:
df.to_csv("../data/wiki_movie_plots_clean.csv", index=False)
print("Saved cleaned dataset to '../data/wiki_movie_plots_clean.csv'")

Saved cleaned dataset to '../data/wiki_movie_plots_clean.csv'
